# ZnO–CO2 analysis of a simulation run

Figures and tables of one run in `results/simulations/<RUN_ID>/`, finished or still
running. Every cell calls the same Python module a user would type in a terminal
(`python -m scripts....`, with `QML_ZNO_RESULTS_DIR=<run dir>`); the notebook only
selects the run and shows the files the modules write. No analysis logic lives here,
so this notebook and the command line give the same results.

Run the cells from `notebooks/` or the repository root. Nothing runs until a
`RUN_*` flag in the configuration cell is set to `True`.

Where the outputs go:

| Step | Writes into |
| --- | --- |
| CO2RR corrections | the run directory (`co2rr_corrected_summary.csv`, `co2rr_corrections_note.md`) |
| CO2RR pathway plot | `results/analyses/<OUTPUT_ID>/co2rr/` |
| CO2 reduction tables | `results/analyses/<OUTPUT_ID>/science/co2_reduction/` |
| NEB table and plots | the run directory (`neb/<material>/comparison_plots/`); the `paper` section writes them to `results/analyses/<OUTPUT_ID>_paper/science/neb/` |
| MD trajectory analysis | the run directory (`md_trajectory_analysis_summary.csv`, `md_desorption_sensitivity.csv`, `md/<material>/<site>/md_frame_analysis.csv`) |
| Publication figures | `results/analyses/<OUTPUT_ID>/science/figures/` |
| Paper figures and tables | `results/analyses/<OUTPUT_ID>_paper/` (`paper/` plus `science/`, with `run_manifest.json`) |

## Local execution

Interactive: open Jupyter in `notebooks/` with an environment that has ASE, pandas
and matplotlib (`qml-zno-simulation`, or `gpaw-dft` on the lab PC), set the flags and
run all cells. Headless:

```bash
cd notebooks
QML_ZNO_SIM_ENV=gpaw-dft ./run_zno_notebook.sh zno-co2-analysis.ipynb zno-co2-analysis.executed.ipynb
```

In [ ]:
from pathlib import Path
import subprocess
import sys
import os
import csv

CWD = Path.cwd()
NOTEBOOK_DIR = next(
    (path for path in [CWD, CWD / "notebooks", CWD.parent / "notebooks"] if (path / "zno-co2-analysis.ipynb").exists()),
    CWD,
)
sim_candidates = [
    CWD,
    CWD.parent,
    NOTEBOOK_DIR.parent,
]
LUCAS_DIR = next((path for path in sim_candidates if (path / "scripts" / "workflows" / "run_simulations.py").exists()), None)
if LUCAS_DIR is None:
    raise FileNotFoundError(
        "Could not resolve simulation scripts. Run from notebooks/ or the repository root."
    )

print(f"Current directory: {CWD}")
print(f"Notebook directory: {NOTEBOOK_DIR}")
print(f"Simulator directory: {LUCAS_DIR}")

def run_cmd(args: list[str], env: dict | None = None) -> None:
    """Run a command from the repository root and stream its output into the cell."""
    print("$", " ".join(args), flush=True)
    with subprocess.Popen(args, cwd=LUCAS_DIR, env=env, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
    if process.returncode:
        raise subprocess.CalledProcessError(process.returncode, args)

def show_table(path: Path, columns: list[str] | None = None, **filters) -> None:
    """Print a CSV written by a module; rows can be filtered by column=value."""
    if not path.exists():
        print(f"[missing] {path}")
        return
    with path.open("r", newline="", encoding="utf-8") as handle:
        rows = [row for row in csv.DictReader(handle)
                if all(row.get(key) == value for key, value in filters.items())]
    print(f"== {path} ({len(rows)} rows)")
    if not rows:
        return
    keep = [name for name in (columns or list(rows[0])) if name in rows[0]]
    try:
        import pandas as pd
        with pd.option_context("display.width", 200, "display.max_columns", None, "display.max_rows", 200):
            print(pd.DataFrame(rows)[keep].to_string(index=False))
    except ImportError:
        print(" | ".join(keep))
        for row in rows:
            print(" | ".join(str(row.get(name, "")) for name in keep))

def list_files(directory: Path, patterns=("*.png", "*.pdf", "*.svg", "*.csv", "*.md")) -> None:
    """List the files a module wrote (figures and tables)."""
    if not directory.is_dir():
        print(f"[missing] {directory}")
        return
    found = sorted({path for pattern in patterns for path in directory.rglob(pattern)})
    for path in found:
        print(f"[OK] {path}")
    if not found:
        print(f"[empty] {directory}")

## Run Configuration

Edit only this cell. The simulation settings were fixed by the run itself
(`config.yaml` in the run directory, copied from `configs/simulations/zno_production.yaml`);
the values here only select what to analyse and where to write it.

In [ ]:
RUN_ID = "zno_ic_final"                 # results/simulations/<RUN_ID>/
OUTPUT_ID = f"{RUN_ID}_notebook"        # results/analyses/<OUTPUT_ID>/
MATERIAL = "ZnO"
CO2RR_VARIANT = "c"                     # a: as computed, b: quasi-harmonic floor, c: b + PBE gas corrections
NEB_BARRIER_RESOLUTION = None           # eV; None: methods.neb.barrier_resolution_ev of the run (0.03)
FIGURE_DPI = 600                        # methods.science_figures.dpi of configs/analysis/zno_current.yaml

RUN_STATUS  = False   # Read-only status dashboard of the run
RUN_CO2RR   = False   # CO2RR corrections, pathway plot and CO2 reduction tables
RUN_NEB     = False   # NEB energy table, barriers and band plots
RUN_MD      = False   # MD trajectory analysis (site occupancy, desorption)
RUN_FIGURES = False   # Publication figures (needs the CO2 reduction tables of RUN_CO2RR)
RUN_PAPER   = False   # Paper figures and tables (run_analysis paper, configs/analysis/zno_paper.yaml)

RUN_DIR = LUCAS_DIR / "results" / "simulations" / RUN_ID
ANALYSIS_DIR = LUCAS_DIR / "results" / "analyses" / OUTPUT_ID
NEB_DIR = RUN_DIR / "neb" / MATERIAL
POST_ENV = {**os.environ, "QML_ZNO_RESULTS_DIR": str(RUN_DIR)}

def run_post(module: str, *args: str) -> None:
    """Analysis module on this run (it reads QML_ZNO_RESULTS_DIR)."""
    run_cmd([sys.executable, "-m", module, *args], env=POST_ENV)

print(f"Run directory: {RUN_DIR}" + ("" if RUN_DIR.is_dir() else "  [missing]"))
print(f"Analysis directory: {ANALYSIS_DIR}")

## 1) Run status

Read-only dashboard of the run (methods, sites, progress, stale calculations).
Same as `python -m scripts.workflows.run_status <RUN_ID>`; it writes nothing.

In [ ]:
if RUN_STATUS:
    run_post("scripts.workflows.run_status", RUN_ID, "--no-color")

## 2) CO2RR pathway and CO2 reduction tables

1. `co2rr_corrections` recomputes the CHE free energies of the run in three variants
   (a, b, c) and writes `co2rr_corrected_summary.csv` and `co2rr_corrections_note.md`
   **into the run directory**, where the next two steps read them.
2. `plot_co2rr_pathway` draws the CO2(g) -> COOH* -> CO* -> CO(g) profile of the chosen variant.
3. `analysis_co2_reduction` merges DFT, TDDFT and CO2RR into the figure-ready tables.

Sites whose intermediates desorb or change identity get no limiting potential.
The site ordering is not resolved when differences are within ~0.05 eV in E_ads
or ~0.1 V in U_L.

In [ ]:
if RUN_CO2RR:
    run_post("scripts.analysis.co2rr_corrections", "--results-dir", str(RUN_DIR))
    run_post(
        "scripts.visualization.plot_co2rr_pathway",
        "--summary", str(RUN_DIR / "co2rr_pathway_summary.csv"),
        "--variant", CO2RR_VARIANT, "--material", MATERIAL,
        "--output-dir", str(ANALYSIS_DIR / "co2rr"),
    )
    run_post(
        "scripts.analysis.analysis_co2_reduction",
        "--results-dir", str(RUN_DIR),
        "--output-dir", str(ANALYSIS_DIR / "science" / "co2_reduction"),
        "--co2rr-variant", CO2RR_VARIANT,
    )

In [ ]:
if RUN_CO2RR:
    show_table(RUN_DIR / "co2rr_corrected_summary.csv",
               ["site", "status", "dG_CO2_to_COOH_ev", "dG_COOH_to_CO_ev", "dG_CO_desorption_ev",
                "limiting_potential_v"],
               material=MATERIAL, variant=CO2RR_VARIANT)
    show_table(ANALYSIS_DIR / "science" / "co2_reduction" / "figure_site_ranking.csv")
    list_files(ANALYSIS_DIR / "co2rr")
    list_files(ANALYSIS_DIR / "science" / "co2_reduction")

## 3) NEB

`neb_energy_table` recomputes the barriers from the stored band energies
(`neb_final_images.traj`); `plot_neb` draws each band (with the force-based spline)
and the barrier comparison. Both write **into the run directory**
(`neb/<MATERIAL>/comparison_plots/`).

Barriers below the resolution (0.03 eV), and bands whose endpoints are not minima
along the band, are reported without a barrier.

In [ ]:
if RUN_NEB:
    resolution = [] if NEB_BARRIER_RESOLUTION is None else ["--barrier-resolution", str(NEB_BARRIER_RESOLUTION)]
    run_post("scripts.analysis.neb_energy_table", "--material", MATERIAL, "--base", str(NEB_DIR), *resolution)
    run_post("scripts.visualization.plot_neb", "--material", MATERIAL, "--base", str(NEB_DIR),
             *resolution, "--smooth-band")

In [ ]:
if RUN_NEB:
    show_table(NEB_DIR / "comparison_plots" / "neb_barriers.csv")
    show_table(NEB_DIR / "comparison_plots" / "neb_summary_table.csv")
    list_files(NEB_DIR / "comparison_plots", patterns=("*.png",))

## 4) Molecular dynamics — trajectory analysis

Per trajectory (site, seed): time at the starting site, lateral displacement and
desorption. Desorption is reported with the CO2-based persistent metric
(`fraction_production_desorbed_persistent`: every CO2 atom farther than the threshold
from the slab for several consecutive frames); `md_desorption_sensitivity.csv` repeats
it for 4.0, 4.5 and 5.0 A. The script writes **into the run directory**, like the
MD section of `zno-co2-lucas.ipynb`; it can run on a partial MD run (partial
trajectories are flagged).

In [ ]:
if RUN_MD:
    run_post("scripts.analysis.md_trajectory_analysis", "--results-dir", str(RUN_DIR), "--material", MATERIAL)

In [ ]:
if RUN_MD:
    show_table(RUN_DIR / "md_trajectory_analysis_summary.csv",
               ["site", "seed", "status", "production_time_fs", "fraction_production_at_start_site",
                "fraction_production_desorbed_persistent", "final_lateral_displacement_ang",
                "mean_c_o_min_slab_distance_ang"])
    show_table(RUN_DIR / "md_desorption_sensitivity.csv")
    plot = RUN_DIR / "md" / MATERIAL / "md_nearest_site_occupancy.png"
    print(f"[OK] {plot}" if plot.exists() else f"[missing] {plot}")

## 5) Publication figures

`render_publication_figures` draws fig01–fig05 (PNG, PDF, SVG) from the CO2 reduction
tables of section 2 and the TDDFT transitions of the run (`tddft/<material>/<site>/transitions.csv`).

The TDDFT spectra are of the CO2 molecule in its adsorbed geometry, isolated in a box
without the slab: a gas-phase reference, not the optical response of ZnO-CO2.
Transitions above the Kohn-Sham ionization threshold (-eps_HOMO) are discretized
continuum states.

Equivalent workflow command (with a manifest and logs; it refuses an existing
`--run-id`, so use a different ID from `OUTPUT_ID`):

```bash
python -m scripts.workflows.run_analysis co2_reduction science_figures \
    --scientific-results results/simulations/<RUN_ID> --run-id <analysis-id>
```

In [ ]:
if RUN_FIGURES:
    co2_tables = ANALYSIS_DIR / "science" / "co2_reduction"
    if not (co2_tables / "co2_reduction_table.csv").exists():
        print(f"[missing] {co2_tables / 'co2_reduction_table.csv'} (run section 2 first)")
    else:
        run_post(
            "scripts.visualization.render_publication_figures",
            "--analysis-dir", str(co2_tables),
            "--tddft-dir", str(RUN_DIR / "tddft"),
            "--output-dir", str(ANALYSIS_DIR / "science" / "figures"),
            "--dpi", str(FIGURE_DPI),
        )
        list_files(ANALYSIS_DIR / "science" / "figures", patterns=("*.png",))

## 6) Paper figures and tables

Runs the `paper` alias of the analysis workflow with `configs/analysis/zno_paper.yaml`
(CO2RR variant c, NEB resolution of the run, TDDFT sigma 0.10 eV, MD seed 1234 in fig06,
600 dpi). The YAML,
not the values of the configuration cell, sets these options. Everything is written to
a new analysis run `results/analyses/<OUTPUT_ID>_paper/`, independent of sections 2-5;
an existing ID is refused.

`paper/` holds `fig01_adsorption_structures.png` (rendered by `export_pymol`: PyMOL when
`import pymol` works, else the ASE/matplotlib fallback; `pymol-open-source` is optional),
`fig01_pymol/` (structures, panels and the PyMOL script of Fig. 1:
`pymol fig01_adsorption_structures.pml`), fig03-fig07 (simple draft style, each with a
`_data.csv`; fig07 is the chained NEB path top metal → bridge → top oxygen) and table02
under their manuscript names and
`figure_registry.csv` (item, file, producer, inputs, sha256; skipped or diagnostic files
are noted). `run_manifest.json` records the Git commit, the hash of the simulation run
manifest and of every input. Same as:

```bash
python -m scripts.workflows.run_analysis paper --config configs/analysis/zno_paper.yaml \
    --scientific-results results/simulations/<RUN_ID> --run-id <OUTPUT_ID>_paper
```


In [ ]:
if RUN_PAPER:
    PAPER_ID = f"{OUTPUT_ID}_paper"
    run_cmd([
        sys.executable, "-m", "scripts.workflows.run_analysis", "paper",
        "--config", str(LUCAS_DIR / "configs" / "analysis" / "zno_paper.yaml"),
        "--scientific-results", str(RUN_DIR),
        "--run-id", PAPER_ID,
    ], env=POST_ENV)
    paper_dir = LUCAS_DIR / "results" / "analyses" / PAPER_ID / "paper"
    show_table(paper_dir / "figure_registry.csv", ["item", "file", "producer_module", "note"])
    list_files(paper_dir, patterns=("*.png", "*.pdf", "*.csv", "*.md", "*.tex"))
